# 🔲 Convolution Benchmark: ROCm vs CPU

Convolutions are inherently GPU-friendly and less optimised by AMD's AVX-512,
making them a workload where MPS can shine even on integrated RDNA 3.5 GPUs.

In [ ]:
import torch
from metalcheck import device_info, is_rocm_available
from metalcheck.utils import benchmark_conv2d

## 1. Device & System Information

In [ ]:
info = device_info()
for k, v in info.items():
    label = k.replace("_", " ").title()
    print(f"{label:<20s}: {v}")

## 2. Conv2d Benchmark

Benchmark standalone `Conv2d` at various spatial sizes, kernel sizes, and channel counts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(style="whitegrid")

spatial_sizes = [64, 128, 256, 512]
kernel_sizes = [3, 5, 7]
channels = 64  # fixed for the main comparison
results = []

for spatial in spatial_sizes:
    for ks in kernel_sizes:
        for dev_name, dev in [("ROCm", torch.device("cuda")), ("CPU", torch.device("cpu"))]:
            if dev_name == "ROCm" and not is_rocm_available():
                continue
            print(f"  spatial={spatial} kernel={ks} | {dev_name} ...", end=" ", flush=True)
            r = benchmark_conv2d(
                spatial_size=spatial,
                in_channels=channels,
                out_channels=channels,
                kernel_size=ks,
                batch_size=1,
                device=dev,
                iterations=10,
            )
            results.append({
                "Spatial": f"{spatial}x{spatial}",
                "Kernel": ks,
                "Device": dev_name,
                "Time (ms)": r["mean_time_s"] * 1000,
            })
            print(f"{r['mean_time_s']*1000:.2f} ms")

df = pd.DataFrame(results)
print("\nConv2d benchmark complete!")

## 3. Visualization

In [ ]:
fig, axes = plt.subplots(1, len(kernel_sizes), figsize=(16, 5), sharey=False)

for ax, ks in zip(axes, kernel_sizes):
    df_ks = df[df["Kernel"] == ks]
    spatial_labels = [f"{s}x{s}" for s in spatial_sizes]
    x = range(len(spatial_sizes))
    width = 0.35

    rocm_vals = df_ks[df_ks["Device"] == "ROCm"]["Time (ms)"].tolist()
    cpu_vals = df_ks[df_ks["Device"] == "CPU"]["Time (ms)"].tolist()

    if rocm_vals:
        ax.bar([i - width / 2 for i in x], rocm_vals, width, label="ROCm (Radeon 890M)", color="#E4002B")
    ax.bar([i + width / 2 for i in x], cpu_vals, width, label="CPU", color="#4A90D9")

    ax.set_xlabel("Input Size")
    ax.set_ylabel("Time (ms)")
    ax.set_title(f"Conv2d — kernel {ks}x{ks}")
    ax.set_xticks(list(x))
    ax.set_xticklabels(spatial_labels)
    ax.legend()

plt.suptitle(f"Conv2d ({channels} channels): ROCm vs CPU", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Channel Scaling

How does increasing the number of channels affect ROCm vs CPU at a fixed spatial size?

In [ ]:
channel_counts = [32, 64, 128, 256]
fixed_spatial = 128
fixed_kernel = 3
channel_results = []

for ch in channel_counts:
    for dev_name, dev in [("ROCm", torch.device("cuda")), ("CPU", torch.device("cpu"))]:
        if dev_name == "ROCm" and not is_rocm_available():
            continue
        print(f"  channels={ch} | {dev_name} ...", end=" ", flush=True)
        r = benchmark_conv2d(
            spatial_size=fixed_spatial,
            in_channels=ch,
            out_channels=ch,
            kernel_size=fixed_kernel,
            batch_size=1,
            device=dev,
            iterations=10,
        )
        channel_results.append({
            "Channels": ch,
            "Device": dev_name,
            "Time (ms)": r["mean_time_s"] * 1000,
        })
        print(f"{r['mean_time_s']*1000:.2f} ms")

df_ch = pd.DataFrame(channel_results)
print("\nChannel scaling benchmark complete!")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x = range(len(channel_counts))
width = 0.35

rocm_vals = df_ch[df_ch["Device"] == "ROCm"]["Time (ms)"].tolist()
cpu_vals = df_ch[df_ch["Device"] == "CPU"]["Time (ms)"].tolist()

if rocm_vals:
    ax.bar([i - width / 2 for i in x], rocm_vals, width, label="ROCm (Radeon 890M)", color="#E4002B")
ax.bar([i + width / 2 for i in x], cpu_vals, width, label="CPU", color="#4A90D9")

ax.set_xlabel("Channels")
ax.set_ylabel("Time (ms)")
ax.set_title(f"Conv2d {fixed_kernel}x{fixed_kernel} @ {fixed_spatial}x{fixed_spatial}: Channel Scaling")
ax.set_xticks(list(x))
ax.set_xticklabels([str(c) for c in channel_counts])
ax.legend()

plt.tight_layout()
plt.show()

## 5. Speedup Summary

In [ ]:
if is_rocm_available():
    rows = []
    for spatial in spatial_sizes:
        for ks in kernel_sizes:
            label = f"{spatial}x{spatial}"
            rocm_t = df[(df["Spatial"] == label) & (df["Kernel"] == ks) & (df["Device"] == "ROCm")]["Time (ms)"].values[0]
            cpu_t = df[(df["Spatial"] == label) & (df["Kernel"] == ks) & (df["Device"] == "CPU")]["Time (ms)"].values[0]
            rows.append({"Input": label, "Kernel": f"{ks}x{ks}", "ROCm (ms)": f"{rocm_t:.2f}", "CPU (ms)": f"{cpu_t:.2f}", "Speedup": f"{cpu_t / rocm_t:.1f}x"})
    display(pd.DataFrame(rows))
else:
    print("ROCm not available — ran CPU only.")